# Feature EDA — Real-Time Bus Monitor

Menganalisis fitur yang digunakan untuk memprediksi `travel_time_sec` antar halte bus WMATA DC.

**Sumber data:**
- `stop_times` + `stops` + `trips` dari PostgreSQL (`batch_data`)
- Semua fitur dihitung ulang dari nol (self-contained)

**Tujuan:**
- Memahami distribusi dan hubungan tiap fitur dengan target
- Evaluasi fitur baru potensial: `hour_of_day`, `stop_position_pct`
- Dasar keputusan untuk feature selection final

In [ ]:
import os
import warnings

import polars as pl
import psycopg2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="Set2")
plt.rcParams["figure.dpi"] = 120
plt.rcParams["figure.figsize"] = (10, 5)

print("All imports OK")

In [ ]:
conn = psycopg2.connect(
    host=os.getenv("PG_HOST", "localhost"),
    port=int(os.getenv("PG_PORT", "5432")),
    dbname=os.getenv("PG_DB", "batch_data"),
    user=os.getenv("PG_USER", "kelompok8"),
    password=os.getenv("PG_PASS", "kelompok8"),
)
print("PostgreSQL connected")

---
## 1. Load & Compute All Features dari PostgreSQL

Pipeline lengkap: join stop_times ↔ stops ↔ trips → hitung jarak antar halte → hitung fitur baru → cleaning → sampling.

In [ ]:
df = pl.read_database(
    """
    SELECT
        st.trip_id,
        st.stop_id,
        st.stop_sequence,
        st.arrival_time,
        s.stop_lat,
        s.stop_lon,
        t.route_id,
        t.direction_id,
        COUNT(*) OVER (PARTITION BY st.trip_id) AS total_stops
    FROM stop_times st
    JOIN stops s ON st.stop_id = s.stop_id
    JOIN trips t ON st.trip_id = t.trip_id
    """,
    conn,
)
conn.close()
print(f"Loaded {len(df):,} rows")

In [ ]:
# Sort untuk perhitungan lag/lead
df = df.sort(["trip_id", "stop_sequence"])

# --- Distance to next stop ---
df = df.with_columns(
    pl.col("stop_lat").shift(-1).over("trip_id").alias("lat_next"),
    pl.col("stop_lon").shift(-1).over("trip_id").alias("lon_next"),
)
df = df.with_columns(lat_mid=(pl.col("stop_lat") + pl.col("lat_next")) / 2)
df = df.with_columns(
    dx=(pl.col("lon_next") - pl.col("stop_lon")) * 111320
       * pl.col("lat_mid").radians().cos(),
    dy=(pl.col("lat_next") - pl.col("stop_lat")) * 111320,
).with_columns(distance_to_next_m=(pl.col("dx") ** 2 + pl.col("dy") ** 2).sqrt())

# --- Travel time ---
def time_to_sec(col):
    parts = col.str.split(":")
    return (
        parts.list.get(0).cast(pl.Int64) * 3600
        + parts.list.get(1).cast(pl.Int64) * 60
        + parts.list.get(2).cast(pl.Int64)
    )

df = df.with_columns(arrival_sec=time_to_sec(pl.col("arrival_time")))
df = df.with_columns(
    arrival_next_sec=pl.col("arrival_sec").shift(-1).over("trip_id"),
).with_columns(travel_time_sec=pl.col("arrival_next_sec") - pl.col("arrival_sec"))

# --- Hour of day ---
df = df.with_columns(hour_of_day=(pl.col("arrival_sec") / 3600).floor().cast(pl.Int64))

# --- Stop position % ---
df = df.with_columns(
    stop_position_pct=pl.col("stop_sequence").cast(pl.Float64) / pl.col("total_stops")
)

print("Features computed")

In [ ]:
# --- Speed estimate (for cleaning) ---
df = df.with_columns(
    speed_est_mps=pl.col("distance_to_next_m") / pl.col("travel_time_sec")
)

# --- Cleaning ---
before = len(df)
df = df.drop_nulls(["distance_to_next_m", "travel_time_sec", "speed_est_mps"])
df = df.filter(
    (pl.col("travel_time_sec") > 0)
    & (pl.col("speed_est_mps") < 30)
    & (pl.col("distance_to_next_m") < 5000)
)
print(f"Cleaned: {before:,} → {len(df):,} ({before - len(df):,} removed)")

In [ ]:
# --- Sample 500K ---
if len(df) > 500_000:
    df = df.sample(n=500_000, seed=42)
    print(f"Sampled to {len(df):,}")

FEATURE_COLS = ["distance_to_next_m", "stop_sequence", "hour_of_day", "stop_position_pct", "route_id"]
TARGET = "travel_time_sec"

---
## 2. Data Overview

In [ ]:
df[FEATURE_COLS + [TARGET]].describe()

In [ ]:
print(f"Nulls:\n{df[FEATURE_COLS + [TARGET]].null_count()}")

---
## 3. Distribusi Tiap Fitur

In [ ]:
numeric_cols = ["distance_to_next_m", "stop_sequence", "travel_time_sec", "hour_of_day", "stop_position_pct"]
titles = ["Distance (m)", "Stop Sequence", "Travel Time (s)", "Hour of Day", "Stop Position %"]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for ax, col, title in zip(axes, numeric_cols, titles):
    arr = df[col].drop_nulls().to_numpy()
    ax.hist(arr, bins=80, edgecolor="none", alpha=0.7)
    ax.axvline(np.median(arr), color="red", ls="--", lw=1, label=f"med={np.median(arr):.0f}")
    ax.axvline(np.mean(arr), color="orange", ls=":", lw=1, label=f"mean={np.mean(arr):.0f}")
    ax.set_title(title)
    ax.legend(fontsize=8)
    ax.yaxis.set_major_formatter(ticker.EngFormatter())

axes[-1].axis("off")
fig.suptitle("Feature Distributions", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---
## 4. Pairwise Scatter

In [ ]:
sample = df.sample(n=10000, seed=42)

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

pairs = [
    ("distance_to_next_m", "travel_time_sec", "Distance (m)", "Travel Time (s)"),
    ("stop_sequence", "travel_time_sec", "Stop Sequence", "Travel Time (s)"),
    ("hour_of_day", "travel_time_sec", "Hour of Day", "Travel Time (s)"),
    ("stop_position_pct", "travel_time_sec", "Stop Position %", "Travel Time (s)"),
]

for ax, (xcol, ycol, xlab, ylab) in zip(axes.flatten(), pairs):
    valid = sample.select(xcol, ycol).drop_nulls()
    ax.scatter(valid[xcol], valid[ycol], alpha=0.3, s=5, c="steelblue")
    ax.set_xlabel(xlab)
    ax.set_ylabel(ylab)

plt.tight_layout()
plt.show()

---
## 5. Korelasi Pearson

In [ ]:
corr_rows = []
for feat in ["distance_to_next_m", "stop_sequence", "hour_of_day", "stop_position_pct"]:
    clean = df.select(feat, TARGET).drop_nulls()
    r = clean.select(pl.corr(feat, TARGET)).item()
    corr_rows.append({"feature": feat, "pearson_r": round(r, 4)})

corr_df = pl.DataFrame(corr_rows).sort("pearson_r", descending=True)
print(corr_df)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

feat_names = [r["feature"] for r in corr_rows]
r_vals = [r["pearson_r"] for r in corr_rows]
colors = ["#2ecc71" if abs(v) > 0.5 else "#f39c12" if abs(v) > 0.1 else "#e74c3c"
          for v in r_vals]

bars = ax.barh(feat_names, r_vals, color=colors, edgecolor="white")
ax.axvline(0, color="black", lw=0.5)
ax.set_xlabel("Pearson r")
ax.set_title("Pearson Correlation with Travel Time", fontsize=13)
ax.set_xlim(-0.2, 0.8)

for bar, r_val in zip(bars, r_vals):
    ax.text(r_val + 0.01 if r_val >= 0 else r_val - 0.05,
            bar.get_y() + bar.get_height() / 2,
            f"{r_val:.4f}", va="center", fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
# Heatmap semua fitur numerik
fig, ax = plt.subplots(figsize=(6, 5))
corr_matrix = df.select(numeric_cols).to_pandas().corr()
sns.heatmap(corr_matrix, annot=True, fmt=".4f", cmap="RdBu_r",
            vmin=-1, vmax=1, center=0, ax=ax, square=True)
ax.set_title("Pearson Correlation Matrix", fontsize=13)
plt.tight_layout()
plt.show()

---
## 6. Hour of Day — Bucket Analysis

Pearson r kecil karena pola non-linear (sinusoidal). Cek per-jam.

In [ ]:
hour_stats = df.group_by("hour_of_day").agg([
    pl.len().alias("count"),
    pl.col("travel_time_sec").mean().round(1).alias("mean_tt"),
    pl.col("travel_time_sec").std().round(1).alias("std_tt"),
    pl.col("travel_time_sec").median().alias("median_tt"),
]).sort("hour_of_day").filter(pl.col("hour_of_day").is_not_null())

print(hour_stats)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))

h_data = hour_stats.filter(pl.col("hour_of_day").between(0, 23)).sort("hour_of_day")

ax.plot(h_data["hour_of_day"], h_data["mean_tt"], marker="o", linewidth=2, color="steelblue")
ax.fill_between(
    h_data["hour_of_day"],
    h_data["mean_tt"] - h_data["std_tt"],
    h_data["mean_tt"] + h_data["std_tt"],
    alpha=0.2, color="steelblue",
)
ax.set_xlabel("Hour of Day")
ax.set_ylabel("Mean Travel Time (s)")
ax.set_title("Travel Time by Hour of Day", fontsize=13)
ax.set_xticks(range(0, 24))
ax.grid(axis="x", alpha=0.3)

for _, row in h_data.iter_rows(named=True):
    ax.annotate(f"{row['mean_tt']:.0f}s",
                (row["hour_of_day"], row["mean_tt"]),
                textcoords="offset points", xytext=(0, 10),
                ha="center", fontsize=8, color="gray")

plt.tight_layout()
plt.show()

**Observasi:**
- Travel time naik dari ~52s (jam 0-5) ke puncak ~77s (jam 16-17) — delta **+48%**
- Pola **non-linear** (sinusoidal), makanya Pearson r kecil meskipun efeknya jelas
- RandomForest bisa menangkap pola ini via split threshold pada jam

---
## 7. Stop Position % — Bucket Analysis

In [ ]:
df_pos = df.with_columns(
    pct_bucket=(pl.col("stop_position_pct") * 10).floor() * 10
)

pos_stats = df_pos.group_by("pct_bucket").agg([
    pl.len().alias("count"),
    pl.col("travel_time_sec").mean().round(1).alias("mean_tt"),
    pl.col("travel_time_sec").std().round(1).alias("std_tt"),
]).sort("pct_bucket")

print(pos_stats)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

p_data = pos_stats.filter(pl.col("pct_bucket").is_not_null())

ax.bar(p_data["pct_bucket"].to_list(), p_data["mean_tt"].to_list(),
       width=8, alpha=0.7, color="steelblue", edgecolor="white")
ax.set_xlabel("Stop Position Percentile")
ax.set_ylabel("Mean Travel Time (s)")
ax.set_title("Travel Time by Stop Position", fontsize=13)
ax.set_xticks(range(0, 110, 10))
ax.set_xticklabels([f"{i}%" for i in range(0, 110, 10)])

for _, row in p_data.iter_rows(named=True):
    ax.text(row["pct_bucket"], row["mean_tt"] + 1,
            f"{row['mean_tt']:.0f}s", ha="center", fontsize=8, color="gray")

plt.tight_layout()
plt.show()

**Observasi:**
- 0-10% (awal trip) jauh lebih tinggi (86s vs 60-69s) — efek departure
- Setelah 10%, relatif flat
- Pola non-linear — tree model bisa split `stop_position_pct <= 0.1`

---
## 8. Route ID Analysis

In [ ]:
route_stats = df.group_by("route_id").agg([
    pl.len().alias("count"),
    pl.col("travel_time_sec").mean().round(1).alias("mean_tt"),
    pl.col("travel_time_sec").std().round(1).alias("std_tt"),
]).sort("mean_tt", descending=True).filter(pl.col("count") > 1000)

print(f"Routes with >1K samples: {len(route_stats)}")
print(route_stats.head(10))
print("...")
print(route_stats.tail(10))

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

top = route_stats.head(20)
colors = sns.color_palette("viridis", n_colors=len(top))
ax.barh(top["route_id"].to_list(), top["mean_tt"].to_list(), color=colors)
ax.set_xlabel("Mean Travel Time (s)")
ax.set_title("Top 20 Routes by Mean Travel Time", fontsize=13)
ax.invert_yaxis()

for i, (_, row) in enumerate(top.iter_rows(named=True)):
    ax.text(row["mean_tt"] + 1, i, f"{row['mean_tt']:.0f}s (n={row['count']:,})",
            va="center", fontsize=8)

plt.tight_layout()
plt.show()

---
## 9. Kesimpulan

| Fitur | Pearson r | Non-linear? | Rekomendasi |
|---|---|---|---|
| `distance_to_next_m` | **+0.73** | ✓ | ✅ **WAJIB** — strongest predictor |
| `stop_sequence` | -0.08 | ✓ (lemah) | ✅ **PAKAI** — bantu konteks urutan |
| `hour_of_day` | ~0.00 | ✓ **kuat** (sinusoidal, delta +48%) | ✅ **TAMBAHKAN** — efek jam sibuk |
| `stop_position_pct` | ~0.03 | ✓ (first-stop spike) | ✅ **TAMBAHKAN** — bedakan awal trip |
| `route_id` | kategorikal | — | ⚠️ **OPSIONAL** — ada variasi antar rute |

**Catatan:** Pearson rendah untuk fitur non-linear (`hour_of_day`, `stop_position_pct`) bukan berarti fitur tidak berguna. Tree-based model seperti RandomForest bisa menangkap pola sinusoidal dan threshold effects.

**Rekomendasi feature set:**
```python
FEATURE_COLS = [
    "distance_to_next_m",   # existing
    "stop_sequence",        # existing
    "hour_of_day",          # baru — dari real-time timestamp
    "stop_position_pct",    # baru — dari stop_sequence / total_stops
]
```